# Movie Recommender — نسخه جدید

این نوت‌بوک برای دیتاست جدید TMDB ساخته شده و به کدهای قدیمی `cast` و `director` وابسته نیست.

قابلیت‌ها:
- بارگذاری خودکار فایل `TMDB_movie_dataset_v11.csv`
- بررسی و پاک‌سازی داده‌ها
- ساخت `tags` از `genres`، `keywords` و `overview`
- جستجوی همه فیلم‌هایی که عبارت واردشده را در عنوان دارند
- ساخت TF-IDF
- پیدا کردن فیلم‌های مشابه با `NearestNeighbors` بدون ساخت ماتریس عظیم cosine similarity
- نمایش امتیاز شباهت فیلم‌های پیشنهادی


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors


## 1. پیدا کردن و خواندن دیتاست


In [ ]:
DATA_DIR = Path('../data')

# ابتدا نام مورد انتظار را امتحان می‌کنیم.
preferred = DATA_DIR / 'TMDB_movie_dataset_v11.csv'

if preferred.exists():
    DATA_PATH = preferred
else:
    candidates = sorted(DATA_DIR.glob('*TMDB*movie*.csv'))
    if not candidates:
        candidates = sorted(DATA_DIR.glob('*.csv'))
    if not candidates:
        raise FileNotFoundError('هیچ فایل CSV در پوشه data پیدا نشد.')
    DATA_PATH = candidates[0]

print('Dataset:', DATA_PATH)
movies = pd.read_csv(DATA_PATH, low_memory=False)
print('Shape:', movies.shape)


In [ ]:
print(movies.columns.tolist())
movies.head()


## 2. آماده‌سازی ستون‌های موردنیاز

در این نسخه از `cast` و `director` استفاده نمی‌کنیم، چون در دیتاست جدید وجود ندارند.


In [ ]:
required = ['id', 'title', 'overview', 'genres', 'keywords']
missing = [c for c in required if c not in movies.columns]
if missing:
    raise KeyError(f'ستون‌های لازم در دیتاست وجود ندارند: {missing}')

movies_clean = movies.copy()

for col in ['title', 'overview', 'genres', 'keywords']:
    movies_clean[col] = movies_clean[col].fillna('').astype(str)

movies_clean = movies_clean[movies_clean['title'].str.strip().ne('')].copy()
movies_clean = movies_clean.reset_index(drop=True)

print('Clean shape:', movies_clean.shape)
movies_clean[['id', 'title', 'genres', 'keywords', 'overview']].head()


## 3. ساخت `tags`

اطلاعات متنی مهم را برای هر فیلم ترکیب می‌کنیم.


In [ ]:
def clean_text(value):
    value = '' if pd.isna(value) else str(value)
    value = value.lower()
    value = re.sub(r'[^a-z0-9]+', ' ', value)
    return re.sub(r'\s+', ' ', value).strip()

movies_clean['tags'] = (
    movies_clean['genres'] + ' ' +
    movies_clean['keywords'] + ' ' +
    movies_clean['overview']
).map(clean_text)

movies_clean[['title', 'tags']].head()


## 4. حذف فیلم‌هایی که هیچ متن قابل استفاده‌ای ندارند


In [ ]:
movies_clean = movies_clean[movies_clean['tags'].str.len() > 0].reset_index(drop=True)
print('Movies with usable text:', len(movies_clean))


## 5. جستجوی عنوان — همه نتایج، نه فقط 10 نتیجه


In [ ]:
def search_movies(query):
    query = str(query).strip()
    if not query:
        return movies_clean.iloc[0:0].copy()

    mask = movies_clean['title'].str.contains(query, case=False, na=False, regex=False)
    result = movies_clean.loc[mask, ['id', 'title', 'release_date', 'vote_average', 'vote_count']].copy()
    return result.sort_values(['title', 'release_date'], na_position='last')

# نمونه:
search_movies('dark')


## 6. ساخت TF-IDF

برای دیتاست 1.4 میلیون‌تایی، ماتریس کامل `cosine_similarity(tfidf_matrix)` نمی‌سازیم؛ چون بسیار بزرگ است.


In [ ]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    dtype=np.float32,
    sublinear_tf=True
)

tfidf_matrix = tfidf.fit_transform(movies_clean['tags'])
print('TF-IDF shape:', tfidf_matrix.shape)


## 7. مدل پیدا کردن فیلم‌های مشابه


In [ ]:
N_NEIGHBORS = 11

model = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=N_NEIGHBORS,
    n_jobs=-1
)

model.fit(tfidf_matrix)
print('Model is ready.')


## 8. پیدا کردن فیلم مشابه


In [ ]:
def recommend_by_index(movie_index, n=10):
    if movie_index < 0 or movie_index >= len(movies_clean):
        raise IndexError('movie_index خارج از محدوده است.')

    n = max(1, int(n))
    k = min(n + 1, len(movies_clean))

    distances, indices = model.kneighbors(tfidf_matrix[movie_index], n_neighbors=k)

    rows = []
    for distance, idx in zip(distances[0], indices[0]):
        if idx == movie_index:
            continue
        rows.append({
            'title': movies_clean.iloc[idx]['title'],
            'release_date': movies_clean.iloc[idx]['release_date'],
            'similarity': round(float(1 - distance), 4),
            'index': int(idx)
        })
        if len(rows) >= n:
            break

    return pd.DataFrame(rows)


## 9. پیشنهاد فیلم بر اساس نام

اگر چند فیلم با یک کلمه پیدا شوند، همه آنها نمایش داده می‌شوند. سپس می‌توانی شماره ردیف موردنظر را برای پیشنهادها انتخاب کنی.


In [ ]:
def recommend(title_query, n=10, movie_index=None):
    matches = search_movies(title_query)

    if matches.empty:
        print(f'هیچ فیلمی با عنوان شامل «{title_query}» پیدا نشد.')
        return None

    # اگر index مشخص شده باشد، همان فیلم انتخاب می‌شود.
    if movie_index is not None:
        selected_index = int(movie_index)
        if selected_index not in matches.index:
            raise ValueError('movie_index انتخاب‌شده در نتایج جستجو نیست.')
    else:
        # اول تطابق دقیق عنوان را ترجیح می‌دهیم؛ در غیر این صورت اولین نتیجه را انتخاب می‌کنیم.
        exact = movies_clean.index[movies_clean['title'].str.casefold() == str(title_query).strip().casefold()].tolist()
        selected_index = exact[0] if exact else matches.index[0]

    selected_title = movies_clean.loc[selected_index, 'title']
    print(f'Selected movie: {selected_title}')
    return recommend_by_index(int(selected_index), n=n)

# نمونه:
recommend('Inception', 10)


## 10. انتخاب بین چند فیلم با عنوان مشابه


In [ ]:
query = 'dark'
results = search_movies(query)

print(f'تعداد نتایج: {len(results):,}')
results.head(50)


### اگر خواستی فیلم مشخصی را از بین نتایج انتخاب کنی

در ستون `index` خروجی بالا، شماره داخلی فیلم را بردار و در تابع زیر قرار بده.


In [ ]:
# مثال:
# recommend('dark', 10, movie_index=12345)


## 11. بررسی نهایی


In [ ]:
print('Number of movies:', len(movies_clean))
print('TF-IDF shape:', tfidf_matrix.shape)
print('Columns:', movies_clean.columns.tolist())

recommend('Inception', 10)


In [20]:
[name for name in globals() if 'tfidf' in name.lower() or 'vector' in name.lower()]

['TfidfVectorizer', 'tfidf', 'tfidf_matrix']

In [21]:
import joblib

joblib.dump(tfidf, "../tfidf_vectorizer.pkl")
joblib.dump(tfidf_matrix, "../tfidf_matrix.pkl")

print("Files saved successfully!")

Files saved successfully!
